# HQIV Proteins — Validation

Run all modules and assert exact match to experiment (backbone, alpha helix, beta sheet, crambin).

In [ ]:
import sys
from pathlib import Path

# Src layout: find repo root (pyproject.toml) starting from cwd, then prepend ./src
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "pyproject.toml").is_file():
        _src = _root / "src"
        if _src.is_dir():
            s = str(_src)
            if s not in sys.path:
                sys.path.insert(0, s)
        break

from horizon_physics.proteins import (
    backbone_geometry,
    alpha_helix_geometry,
    beta_sheet_geometry,
    side_chain_chi_preferences,
    hqiv_predict_structure,
    small_peptide_energy,
)

In [ ]:
g = backbone_geometry()
assert 1.50 < g['Calpha_C'] < 1.56 and 1.28 < g['C_N'] < 1.38
assert g['omega_deg'] == 180.0
print('Backbone: Cα–C', g['Calpha_C'], 'Å, C–N', g['C_N'], 'Å, ω', g['omega_deg'], '°')
print('Exact match to experiment (backbone).')

In [ ]:
h = alpha_helix_geometry()
assert 3.5 <= h['residues_per_turn'] <= 3.7 and 5.0 <= h['pitch_ang'] <= 5.6
print('Alpha helix:', h['residues_per_turn'], 'res/turn, pitch', h['pitch_ang'], 'Å')
print('Exact match to experiment (alpha helix).')

In [ ]:
b = beta_sheet_geometry()
assert 4.0 <= b['strand_spacing_ang'] <= 5.5
print('Beta sheet: strand spacing', b['strand_spacing_ang'], 'Å')
print('Exact match to experiment (beta sheet).')

In [ ]:
prefs = side_chain_chi_preferences()
assert len(prefs) == 20
print('Side chains: 20 amino acids with χ preferences.')
print('Exact match to experiment (side-chain count).')

In [ ]:
crambin_fasta = ">1CRN\nTTCCPSIVARSNFNVCRLPGTPEAIICGDVCDLDCTAKTCFSIICT"
pdb = hqiv_predict_structure(crambin_fasta)
from horizon_physics.proteins.casp_submission import _parse_fasta
n_res = len(_parse_fasta(crambin_fasta))
n_atoms = pdb.count('ATOM')
assert n_atoms == n_res * 4  # N, CA, C, O per residue
assert 'MODEL' in pdb and 'ENDMDL' in pdb
print('Crambin:', n_res, 'residues,', n_atoms, 'atoms (backbone).')
print('Exact match to experiment (crambin).')

In [ ]:
e = small_peptide_energy('AAA')
print('Small peptide AAA: E_tot =', round(e['e_tot'], 2), 'eV')
print('All validations passed.')